/# IV Scalping Analysis: Mark 01 & Mark 22

Analyzing trading patterns between Mark 01 and Mark 22 to identify IV scalping opportunities,
particularly during low-spread windows and implied volatility reversals in voucher (VEV) products.

**Hypothesis**: When spreads tighten and Mark 01/22 establish asymmetric positions in vouchers,
there may be mean-reverting IV opportunities to exploit through delta-hedged option scalps.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from scipy.optimize import brentq, minimize_scalar
import plotly.graph_objects as go
import plotly.express as px
from typing import Tuple, Optional, List, Dict
import warnings
warnings.filterwarnings('ignore')

# Set seed for reproducibility
np.random.seed(42)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print("✓ All imports successful")

✓ All imports successful


In [2]:
## 1. DATA INGESTION: Load Trades and Prices

# Setup paths
data_dir = "Dataset/ROUND_4"
prices_files = [
    "prices_round_4_day_1.csv",
    "prices_round_4_day_2.csv",
    "prices_round_4_day_3.csv",
]
trades_files = [
    "trades_round_4_day_1.csv",
    "trades_round_4_day_2.csv",
    "trades_round_4_day_3.csv",
]

# Load trades
trades_list = []
for fname in trades_files:
    fpath = os.path.join(data_dir, fname)
    if os.path.exists(fpath):
        df = pd.read_csv(fpath, sep=";")
        day_num = int(fname.split("_")[-1].split(".")[0])
        df['day'] = day_num
        trades_list.append(df)
        print(f"Loaded {fname}: {len(df)} rows")

trades = pd.concat(trades_list, ignore_index=True)
print(f"\nTotal trades: {len(trades)}")
print(f"Trades columns: {trades.columns.tolist()}")
print(f"Sample:\n{trades.head()}")

# Load prices
prices_list = []
for fname in prices_files:
    fpath = os.path.join(data_dir, fname)
    if os.path.exists(fpath):
        df = pd.read_csv(fpath, sep=";")
        day_num = int(fname.split("_")[-1].split(".")[0])
        df['day'] = day_num
        prices_list.append(df)
        print(f"Loaded {fname}: {len(df)} rows")

prices = pd.concat(prices_list, ignore_index=True)
print(f"\nTotal price rows: {len(prices)}")
print(f"Price columns: {prices.columns.tolist()}")
print(f"Sample:\n{prices.head()}")

Loaded trades_round_4_day_1.csv: 1407 rows
Loaded trades_round_4_day_2.csv: 1333 rows
Loaded trades_round_4_day_3.csv: 1541 rows

Total trades: 4281
Trades columns: ['timestamp', 'buyer', 'seller', 'symbol', 'currency', 'price', 'quantity', 'day']
Sample:
   timestamp    buyer   seller         symbol currency     price  quantity  \
0       4500  Mark 01  Mark 22       VEV_5400   XIRECS   18.0000         2   
1       4500  Mark 01  Mark 22       VEV_5500   XIRECS    8.0000         2   
2       4500  Mark 01  Mark 22       VEV_6000   XIRECS    0.0000         2   
3       4500  Mark 01  Mark 22       VEV_6500   XIRECS    0.0000         2   
4       5100  Mark 38  Mark 22  HYDROGEL_PACK   XIRECS 9960.0000         4   

   day  
0    1  
1    1  
2    1  
3    1  
4    1  
Loaded prices_round_4_day_1.csv: 120000 rows
Loaded prices_round_4_day_2.csv: 120000 rows
Loaded prices_round_4_day_3.csv: 120000 rows

Total price rows: 360000
Price columns: ['day', 'timestamp', 'product', 'bid_price_1'

In [3]:
## 2. FILTER MARK 01 & MARK 22 TRADES

# Filter trades involving Mark 01 and Mark 22
mark_01_22_trades = trades[
    ((trades['buyer'] == 'Mark 01') & (trades['seller'] == 'Mark 22')) |
    ((trades['buyer'] == 'Mark 22') & (trades['seller'] == 'Mark 01'))
].copy()

print(f"Trades involving Mark 01 & Mark 22: {len(mark_01_22_trades)}")
print(f"\nTrade breakdown by side:")
print(f"  Mark 01 buying: {len(mark_01_22_trades[mark_01_22_trades['buyer'] == 'Mark 01'])}")
print(f"  Mark 01 selling: {len(mark_01_22_trades[mark_01_22_trades['seller'] == 'Mark 01'])}")

print(f"\nTrade breakdown by product:")
print(mark_01_22_trades['symbol'].value_counts())

print(f"\nSample Mark 01/22 trades:")
print(mark_01_22_trades.head(15))

Trades involving Mark 01 & Mark 22: 1339

Trade breakdown by side:
  Mark 01 buying: 1339
  Mark 01 selling: 0

Trade breakdown by product:
symbol
VEV_6000    317
VEV_6500    317
VEV_5500    299
VEV_5400    263
VEV_5300    132
VEV_5200     11
Name: count, dtype: int64

Sample Mark 01/22 trades:
    timestamp    buyer   seller    symbol currency   price  quantity  day
0        4500  Mark 01  Mark 22  VEV_5400   XIRECS 18.0000         2    1
1        4500  Mark 01  Mark 22  VEV_5500   XIRECS  8.0000         2    1
2        4500  Mark 01  Mark 22  VEV_6000   XIRECS  0.0000         2    1
3        4500  Mark 01  Mark 22  VEV_6500   XIRECS  0.0000         2    1
9        9600  Mark 01  Mark 22  VEV_5400   XIRECS 16.0000         4    1
10       9600  Mark 01  Mark 22  VEV_5500   XIRECS  7.0000         4    1
11       9600  Mark 01  Mark 22  VEV_6000   XIRECS  0.0000         4    1
12       9600  Mark 01  Mark 22  VEV_6500   XIRECS  0.0000         4    1
14      11800  Mark 01  Mark 22  VEV_5

In [4]:
## 3. PREPROCESSING: Compute Spreads and Normalize Prices

# Create a time index combining day and timestamp
prices['time_key'] = prices['day'].astype(str) + "_" + prices['timestamp'].astype(str)
prices['global_timestamp'] = prices['day'] * 10000 + prices['timestamp']

mark_01_22_trades['time_key'] = mark_01_22_trades['day'].astype(str) + "_" + mark_01_22_trades['timestamp'].astype(str)
mark_01_22_trades['global_timestamp'] = mark_01_22_trades['day'] * 10000 + mark_01_22_trades['timestamp']

# Compute bid-ask spreads and mid prices
prices['bid'] = prices['bid_price_1']
prices['ask'] = prices['ask_price_1']
prices['spread'] = prices['ask'] - prices['bid']
prices['mid'] = (prices['bid'] + prices['ask']) / 2

# Filter for voucher products (VEV_*)
voucher_prices = prices[prices['product'].str.startswith('VEV', na=False)].copy()

print(f"Total voucher price rows: {len(voucher_prices)}")
print(f"\nVoucher products available:")
print(voucher_prices['product'].unique())

# Compute spread statistics
print(f"\nSpread statistics (vouchers only):")
print(f"  Mean spread: {voucher_prices['spread'].mean():.4f}")
print(f"  Median spread: {voucher_prices['spread'].median():.4f}")
print(f"  Min spread: {voucher_prices['spread'].min():.4f}")
print(f"  Max spread: {voucher_prices['spread'].max():.4f}")
print(f"  Std spread: {voucher_prices['spread'].std():.4f}")

# Get underlying (VELVETFRUIT_EXTRACT)
underlying_prices = prices[prices['product'] == 'VELVETFRUIT_EXTRACT'].copy()
print(f"\nUnderlying prices: {len(underlying_prices)}")
print(f"Underlying price range: {underlying_prices['mid'].min():.0f} - {underlying_prices['mid'].max():.0f}")

Total voucher price rows: 300000

Voucher products available:
<StringArray>
['VEV_6000', 'VEV_5000', 'VEV_6500', 'VEV_5300', 'VEV_5400', 'VEV_4000',
 'VEV_5100', 'VEV_5200', 'VEV_5500', 'VEV_4500']
Length: 10, dtype: str

Spread statistics (vouchers only):
  Mean spread: 5.5830
  Median spread: 2.0000
  Min spread: 1.0000
  Max spread: 22.0000
  Std spread: 6.6591

Underlying prices: 30000
Underlying price range: 5192 - 5300


In [5]:
## 4. BLACK-SCHOLES UTILITIES

def bs_call_price(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """Black-Scholes call price."""
    if T <= 0 or sigma <= 0:
        return max(S - K, 0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)

def bs_put_price(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """Black-Scholes put price."""
    if T <= 0 or sigma <= 0:
        return max(K - S, 0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

def bs_vega(S: float, K: float, T: float, r: float, sigma: float) -> float:
    """Vega (dPrice/dSigma) per 1% change in vol."""
    if T <= 0 or sigma <= 0:
        return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return S * norm.pdf(d1) * np.sqrt(T) / 100.0

def implied_vol(market_price: float, S: float, K: float, T: float, r: float,
                option_type: str = 'call', tol: float = 1e-6) -> Optional[float]:
    """
    Compute implied volatility using Brent's method.
    Returns None if no valid IV found.
    """
    if option_type == 'call':
        bs_func = bs_call_price
        intrinsic = max(S - K, 0)
    else:
        bs_func = bs_put_price
        intrinsic = max(K - S, 0)
    
    # If market price < intrinsic, return None (arbitrage)
    if market_price < intrinsic * 0.99:
        return None
    
    # Try to bracket the solution
    try:
        # Define objective
        def obj(sigma):
            return bs_func(S, K, T, r, sigma) - market_price
        
        # Search for IV in range [0.001, 3.0]
        iv = brentq(obj, 0.001, 3.0, xtol=tol)
        return iv
    except ValueError:
        # Brent failed to bracket; try wider search
        try:
            iv = brentq(lambda s: bs_func(S, K, T, r, s) - market_price, 
                       0.0001, 5.0, xtol=tol)
            return iv
        except:
            return None

print("✓ Black-Scholes functions defined")

✓ Black-Scholes functions defined


In [6]:
## 5. ANALYZE VOUCHER STRUCTURE & STRIKES

# Extract strike from product name (e.g., VEV_5300 -> strike=5300)
voucher_prices['strike'] = voucher_prices['product'].str.extract(r'(\d+)').astype(int)

# Infer option type from strike relative to initial spot (~5245)
initial_spot = 5245  # Approximate from data
voucher_prices['option_type'] = voucher_prices['strike'].apply(
    lambda k: 'call' if k > initial_spot else 'put'
)

# Time-to-expiry: Round 4 has 4d on day 1, 3d on day 2, 2d on day 3
tte_map = {1: 4/365, 2: 3/365, 3: 2/365}
voucher_prices['tte_years'] = voucher_prices['day'].map(tte_map)

# Get underlying spot for each timestamp
underlying_by_time = underlying_prices.groupby('global_timestamp')['mid'].first()

print("Voucher structure:")
print(voucher_prices[['product', 'strike', 'option_type', 'tte_years', 'day']].drop_duplicates().sort_values('strike'))

print(f"\nStrike range: {voucher_prices['strike'].min()} - {voucher_prices['strike'].max()}")
print(f"Time-to-expiry: {voucher_prices['tte_years'].unique()}")

# Get spot prices at each timestamp
voucher_prices['spot'] = voucher_prices['global_timestamp'].map(underlying_by_time)

print(f"\nSpot price stats:")
print(f"  Mean: {voucher_prices['spot'].mean():.2f}")
print(f"  Range: {voucher_prices['spot'].min():.2f} - {voucher_prices['spot'].max():.2f}")

Voucher structure:
         product  strike option_type  tte_years  day
7       VEV_4000    4000         put     0.0110    1
120003  VEV_4000    4000         put     0.0082    2
240001  VEV_4000    4000         put     0.0055    3
11      VEV_4500    4500         put     0.0110    1
240008  VEV_4500    4500         put     0.0055    3
120005  VEV_4500    4500         put     0.0082    2
120002  VEV_5000    5000         put     0.0082    2
3       VEV_5000    5000         put     0.0110    1
240004  VEV_5000    5000         put     0.0055    3
240002  VEV_5100    5100         put     0.0055    3
8       VEV_5100    5100         put     0.0110    1
120004  VEV_5100    5100         put     0.0082    2
9       VEV_5200    5200         put     0.0110    1
120000  VEV_5200    5200         put     0.0082    2
240003  VEV_5200    5200         put     0.0055    3
120001  VEV_5300    5300        call     0.0082    2
5       VEV_5300    5300        call     0.0110    1
240010  VEV_5300    5300   

In [ ]:
## 6. COMPUTE IMPLIED VOLATILITIES (BID / MID / ASK)

# Apply IV solver to bid, mid, and ask prices
r = 0.0  # Risk-free rate ~0 for Prosperity
print("Computing implied volatilities...")

voucher_prices['iv_bid'] = voucher_prices.apply(
    lambda row: implied_vol(row['bid'], row['spot'], row['strike'], row['tte_years'], r, row['option_type']),
    axis=1
)

voucher_prices['iv_mid'] = voucher_prices.apply(
    lambda row: implied_vol(row['mid'], row['spot'], row['strike'], row['tte_years'], r, row['option_type']),
    axis=1
)

voucher_prices['iv_ask'] = voucher_prices.apply(
    lambda row: implied_vol(row['ask'], row['spot'], row['strike'], row['tte_years'], r, row['option_type']),
    axis=1
)

# Compute IV gaps
voucher_prices['iv_spread'] = voucher_prices['iv_ask'] - voucher_prices['iv_bid']
voucher_prices['iv_mid_bid_gap'] = voucher_prices['iv_mid'] - voucher_prices['iv_bid']

# Remove rows with NaN IVs (arbitrage or pricing errors)
print(f"Rows with valid IV (all 3): {(voucher_prices[['iv_bid', 'iv_mid', 'iv_ask']].notna().all(axis=1)).sum()}")

print(f"\nImplied Volatility Summary:")
print(f"IV Bid   - Mean: {voucher_prices['iv_bid'].mean():.4f}, Range: [{voucher_prices['iv_bid'].min():.4f}, {voucher_prices['iv_bid'].max():.4f}]")
print(f"IV Mid   - Mean: {voucher_prices['iv_mid'].mean():.4f}, Range: [{voucher_prices['iv_mid'].min():.4f}, {voucher_prices['iv_mid'].max():.4f}]")
print(f"IV Ask   - Mean: {voucher_prices['iv_ask'].mean():.4f}, Range: [{voucher_prices['iv_ask'].min():.4f}, {voucher_prices['iv_ask'].max():.4f}]")
print(f"IV Spread- Mean: {voucher_prices['iv_spread'].mean():.4f}, Median: {voucher_prices['iv_spread'].median():.4f}")

# Show by product
print(f"\nIV stats by voucher product:")
iv_by_product = voucher_prices.groupby('product')[['spread', 'iv_bid', 'iv_mid', 'iv_ask', 'iv_spread']].agg(['mean', 'median', 'std'])
print(iv_by_product)

Computing implied volatilities...


In [ ]:
## 7. IDENTIFY LOW-SPREAD WINDOWS & SCALPING CANDIDATES

# Define spread thresholds
spread_percentiles = voucher_prices['spread'].quantile([0.1, 0.25, 0.5, 0.75])
print("Spread percentiles:")
print(spread_percentiles)

# Filter for low-spread opportunities
low_spread_threshold = spread_percentiles[0.25]  # Bottom quartile
tight_spread = voucher_prices[voucher_prices['spread'] <= low_spread_threshold].copy()

print(f"\nLow-spread filter (spread <= {low_spread_threshold:.2f}):")
print(f"  Rows: {len(tight_spread)}")
print(f"  % of total: {100*len(tight_spread)/len(voucher_prices):.1f}%")

# Look for scalping signals:
# 1. Low spread + positive IV mid-bid gap (mid priced above bid)
# 2. Look for reversions: entry when IV_spread is wide, exit when it tightens

# Calculate IV spread percentile to identify when spreads are especially wide
voucher_prices['iv_spread_pctl'] = voucher_prices.groupby('product')['iv_spread'].transform(
    lambda x: x.rank(pct=True) if len(x) > 1 else 0.5
)

# Identify scalping candidates: low price spread + wide IV spread
scalp_candidates = voucher_prices[
    (voucher_prices['spread'] <= low_spread_threshold) & 
    (voucher_prices['iv_spread_pctl'] >= 0.75)
].copy()

print(f"\nScalping candidates (low price spread + wide IV spread):")
print(f"  Rows: {len(scalp_candidates)}")

if len(scalp_candidates) > 0:
    print(f"\nCandidate details:")
    print(scalp_candidates[['product', 'day', 'timestamp', 'spread', 'iv_spread', 'iv_bid', 'iv_mid', 'iv_ask']].head(20))

# Visualize spread vs IV spread relationship
print(f"\nSpread vs IV Spread correlation:")
valid_data = voucher_prices[['spread', 'iv_spread']].dropna()
if len(valid_data) > 1:
    corr = valid_data['spread'].corr(valid_data['iv_spread'])
    print(f"  Correlation: {corr:.4f}")

In [ ]:
## 8. ANALYZE MARK 01 & 22 TRADES AT LOW-SPREAD WINDOWS

# Join Mark trades with prices at nearest timestamp
def find_nearest_price(trade_row):
    """Find closest price tick for a trade."""
    key = f"{trade_row['day']}_"  # Prefix for day
    matching_prices = voucher_prices[
        (voucher_prices['day'] == trade_row['day']) &
        (voucher_prices['product'] == trade_row['symbol'])
    ]
    if len(matching_prices) == 0:
        return None
    
    # Find closest timestamp
    time_diffs = (matching_prices['timestamp'] - trade_row['timestamp']).abs()
    nearest_idx = time_diffs.idxmin()
    return nearest_idx

mark_01_22_trades['price_idx'] = mark_01_22_trades.apply(find_nearest_price, axis=1)

# Merge price data
trade_analysis = mark_01_22_trades.dropna(subset=['price_idx']).copy()
trade_analysis['price_idx'] = trade_analysis['price_idx'].astype(int)

price_cols = ['product', 'spread', 'iv_spread', 'iv_bid', 'iv_mid', 'iv_ask', 'bid', 'ask', 'mid']
for col in price_cols:
    trade_analysis[col + '_at_trade'] = trade_analysis['price_idx'].map(
        voucher_prices.set_index(voucher_prices.index)[col]
    )

trade_analysis = trade_analysis.dropna(subset=['spread_at_trade'])

print(f"Mark 01/22 trades with pricing data: {len(trade_analysis)}")

# Analyze spread conditions at time of Mark trades
print(f"\nSpread conditions during Mark 01/22 trades:")
print(f"  Mean spread: {trade_analysis['spread_at_trade'].mean():.4f}")
print(f"  Median spread: {trade_analysis['spread_at_trade'].median():.4f}")
print(f"  Min spread: {trade_analysis['spread_at_trade'].min():.4f}")
print(f"  Max spread: {trade_analysis['spread_at_trade'].max():.4f}")

# Break down by side
mark_01_buy = trade_analysis[trade_analysis['buyer'] == 'Mark 01']
mark_01_sell = trade_analysis[trade_analysis['seller'] == 'Mark 01']

print(f"\nMark 01 BUYING (from Mark 22):")
print(f"  Trades: {len(mark_01_buy)}")
if len(mark_01_buy) > 0:
    print(f"  Mean spread: {mark_01_buy['spread_at_trade'].mean():.4f}")
    print(f"  Mean IV spread: {mark_01_buy['iv_spread_at_trade'].mean():.4f}")
    print(f"  Products: {mark_01_buy['symbol'].value_counts().to_dict()}")

print(f"\nMark 01 SELLING (to Mark 22):")
print(f"  Trades: {len(mark_01_sell)}")
if len(mark_01_sell) > 0:
    print(f"  Mean spread: {mark_01_sell['spread_at_trade'].mean():.4f}")
    print(f"  Mean IV spread: {mark_01_sell['iv_spread_at_trade'].mean():.4f}")
    print(f"  Products: {mark_01_sell['symbol'].value_counts().to_dict()}")

In [ ]:
## 9. IV MEAN REVERSION: Track IV changes post-trade

# For each Mark trade, look at IV changes in following ticks
def analyze_iv_reversion(trade_row):
    """Analyze IV reversion after a Mark trade."""
    product = trade_row['symbol']
    day = trade_row['day']
    timestamp = trade_row['timestamp']
    
    # Get product history for this day
    product_history = voucher_prices[
        (voucher_prices['product'] == product) &
        (voucher_prices['day'] == day)
    ].sort_values('timestamp')
    
    if len(product_history) < 2:
        return {}
    
    # Find current and future ticks
    current_mask = product_history['timestamp'] == timestamp
    if not current_mask.any():
        # Find closest
        time_diffs = (product_history['timestamp'] - timestamp).abs()
        closest_idx = time_diffs.idxmin()
        current_idx = product_history.index.get_loc(closest_idx)
    else:
        current_idx = product_history.index.get_loc(product_history[current_mask].index[0])
    
    result = {
        'product': product,
        'trade_timestamp': timestamp,
        'iv_mid_at_trade': product_history.iloc[current_idx]['iv_mid'],
        'spread_at_trade': product_history.iloc[current_idx]['spread']
    }
    
    # Look at next 5 ticks
    for offset in range(1, min(6, len(product_history) - current_idx)):
        future_row = product_history.iloc[current_idx + offset]
        result[f'iv_mid_+{offset}'] = future_row['iv_mid']
        result[f'spread_+{offset}'] = future_row['spread']
    
    return result

reversion_data = []
for _, trade in mark_01_22_trades.iterrows():
    reversion_data.append(analyze_iv_reversion(trade))

reversion_df = pd.DataFrame(reversion_data)

print(f"IV Mean Reversion Analysis:")
print(f"Trades analyzed: {len(reversion_df)}")

if len(reversion_df) > 0:
    # Calculate average IV changes
    iv_changes = []
    for offset in range(1, 6):
        col = f'iv_mid_+{offset}'
        if col in reversion_df.columns:
            changes = reversion_df[col] - reversion_df['iv_mid_at_trade']
            iv_changes.append({
                'offset': offset,
                'mean_iv_change': changes.mean(),
                'median_iv_change': changes.median(),
                'pct_positive': (changes > 0).sum() / len(changes) * 100
            })
    
    reversion_summary = pd.DataFrame(iv_changes)
    print(f"\nIV changes (offset in ticks from trade):")
    print(reversion_summary)
    
    # Check spreads at same time
    spread_cols = [f'spread_+{i}' for i in range(1, 6) if f'spread_+{i}' in reversion_df.columns]
    if len(spread_cols) > 0:
        print(f"\nSpread changes after trade:")
        for col in spread_cols:
            spread_changes = reversion_df[col] - reversion_df['spread_at_trade']
            print(f"  {col}: mean={spread_changes.mean():.4f}, median={spread_changes.median():.4f}")

In [ ]:
## 10. VISUALIZATION: Spread vs IV Spread Dynamics

fig = go.Figure()

# Scatter plot: spread vs iv_spread
valid_scatter = voucher_prices[['spread', 'iv_spread', 'product']].dropna()

for product in valid_scatter['product'].unique():
    product_data = valid_scatter[valid_scatter['product'] == product]
    fig.add_trace(go.Scatter(
        x=product_data['spread'],
        y=product_data['iv_spread'],
        mode='markers',
        name=product,
        marker=dict(size=5, opacity=0.6)
    ))

fig.update_layout(
    title="Spread vs IV Spread (Scalping Opportunity Space)",
    xaxis_title="Price Spread (bid-ask)",
    yaxis_title="IV Spread (ask IV - bid IV)",
    hovermode='closest',
    height=500,
    template='plotly_dark'
)
fig.show()

# Histogram of spreads and IV spreads
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(voucher_prices['spread'].dropna(), bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Price Spread')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Price Spreads')
axes[0].axvline(low_spread_threshold, color='red', linestyle='--', label=f'Q1 Threshold ({low_spread_threshold:.2f})')
axes[0].legend()

axes[1].hist(voucher_prices['iv_spread'].dropna(), bins=50, color='coral', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('IV Spread')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of IV Spreads')
axes[1].axvline(voucher_prices['iv_spread'].quantile(0.75), color='red', linestyle='--', label='Q3 Threshold')
axes[1].legend()

plt.tight_layout()
plt.show()

print("✓ Spread distributions plotted")

In [ ]:
## 11. SUMMARY: IV Scalping Opportunities with Mark 01 & 22

print("="*80)
print("IV SCALPING ANALYSIS SUMMARY: MARK 01 & MARK 22")
print("="*80)

print("\n[TRADE ACTIVITY]")
print(f"Total Mark 01 <-> Mark 22 trades: {len(mark_01_22_trades)}")
print(f"  - Mark 01 BUY side:  {(mark_01_22_trades['buyer'] == 'Mark 01').sum()} trades")
print(f"  - Mark 01 SELL side: {(mark_01_22_trades['seller'] == 'Mark 01').sum()} trades")
print(f"Primary products traded: {mark_01_22_trades['symbol'].value_counts().head().to_dict()}")

print("\n[SPREAD ANALYSIS]")
print(f"Median price spread (all vouchers): {voucher_prices['spread'].median():.4f}")
print(f"Q1 spread threshold (for 'tight'): {low_spread_threshold:.4f}")
print(f"% of time spreads are 'tight': {100*(voucher_prices['spread'] <= low_spread_threshold).sum()/len(voucher_prices):.1f}%")

print("\n[IV SPREAD DYNAMICS]")
print(f"Median IV spread (ask-bid): {voucher_prices['iv_spread'].median():.4f}")
print(f"Mean IV spread: {voucher_prices['iv_spread'].mean():.4f}")
print(f"IV spread correlation with price spread: {valid_data['spread'].corr(valid_data['iv_spread']):.4f}")

print("\n[KEY FINDING 1: MARK TRADE TIMING vs SPREADS]")
if len(trade_analysis) > 0:
    print(f"Mark 01 trades occur when spreads are:")
    avg_mark_spread = trade_analysis['spread_at_trade'].mean()
    percentile_of_mark_spread = (voucher_prices['spread'] <= avg_mark_spread).sum() / len(voucher_prices) * 100
    print(f"  Average: {avg_mark_spread:.4f} (at {percentile_of_mark_spread:.0f}th percentile)")
    print(f"  Median:  {trade_analysis['spread_at_trade'].median():.4f}")
    
    if percentile_of_mark_spread < 50:
        print(f"  ✓ Mark 01/22 PREFER TIGHT SPREADS - suggests scalping mindset")
    else:
        print(f"  ✗ Marks trade across spread distribution - not specifically seeking tight spreads")

print("\n[KEY FINDING 2: IV REVERSION AFTER MARK TRADES]")
if len(reversion_summary) > 0:
    avg_5tick_reversion = reversion_summary[reversion_summary['offset'] == 5]['mean_iv_change'].values
    if len(avg_5tick_reversion) > 0:
        print(f"Mean IV change 5 ticks after trade: {avg_5tick_reversion[0]:.6f}")
        win_rate = reversion_summary[reversion_summary['offset'] == 5]['pct_positive'].values
        if len(win_rate) > 0:
            print(f"% of times IV moves HIGHER in next 5 ticks: {win_rate[0]:.1f}%")

print("\n[SCALPING SWEET SPOT CANDIDATE]")
print(f"Low price spread + wide IV spread instances: {len(scalp_candidates)}")
if len(scalp_candidates) > 0:
    print(f"  ✓ YES - {len(scalp_candidates)} instances where both conditions met")
    print(f"  These could be profitable entry points:")
    print(f"    - Entry: Sell volatility when price spread is tight but IV spread is wide")
    print(f"    - Exit: Buy back when IV spread compresses")
    
    sample_trades = scalp_candidates[['product', 'day', 'timestamp', 'spread', 'iv_spread']].head(5)
    print(f"\n  Example scalp windows:")
    for idx, (i, row) in enumerate(sample_trades.iterrows(), 1):
        print(f"    {idx}. Day {row['day']}, ts={row['timestamp']:.0f}, {row['product']}: spread={row['spread']:.2f}, iv_spread={row['iv_spread']:.4f}")
else:
    print(f"  ✗ NO instances - no obvious scalping windows")

print("\n[RECOMMENDATIONS]")
print("1. Monitor Mark 01 trades as leading indicators - they show when to expect market moves")
print("2. Focus on vouchers with natural IV spread (OTM strikes) for vega scalping")
print("3. Entry signal: Tight price spread + Mark 01/22 activity + wide IV spread")
print("4. Exit signal: IV spread mean reversion or adverse spot move >1 delta")
print("5. Risk management: Size inversely to IV spread width (less vol = smaller size)")
print("="*80)